# Интерактивный запуск проекта

Этот ноутбук помогает быстро просмотреть статическую HTML-версию интерфейса и запустить локальную версию Dash-приложения из `app.py`. Пошаговые инструкции включены в отдельных разделах.

## Подготовка окружения

1. Убедитесь, что у вас установлен Python 3.10 или новее.
2. Создайте и активируйте виртуальное окружение:
```
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\\Scripts\\activate
```
3. Установите зависимости проекта:
```
pip install -r requirements.txt
```

## Просмотр статической HTML-версии

Выполните ячейку ниже, чтобы отобразить сохранённую HTML-версию дашборда (`cooperative_rnd_model_interactive.html`).

> **Совет:** если в браузере Jupyter отображение не открывается во встроенном фрейме, скачайте файл напрямую и откройте его в отдельной вкладке.

In [ ]:
from pathlib import Path
from IPython.display import IFrame, display

html_path = Path('cooperative_rnd_model_interactive.html').resolve()
if html_path.exists():
    display(IFrame(src=html_path.as_uri(), width='100%', height=600))
else:
    raise FileNotFoundError(f'HTML-файл не найден: {html_path}')

## Запуск Dash-приложения (`app.py`)

1. Убедитесь, что зависимости установлены (см. раздел выше).
2. Выполните следующую ячейку, чтобы запустить сервер прямо из ноутбука. После запуска откройте адрес [http://127.0.0.1:8050/](http://127.0.0.1:8050/) в браузере.
3. После завершения работы не забудьте остановить сервер, выполнив ячейку **"Остановка сервера"** ниже.

> **Примечание:** ячейка запуска проверяет, не запущен ли сервер ранее, чтобы избежать дубликатов процессов.

In [ ]:
import pathlib
import subprocess
import sys

if 'app_process' in globals() and app_process.poll() is None:
    print('Сервер уже запущен на http://127.0.0.1:8050/.')
else:
    project_root = pathlib.Path.cwd()
    app_process = subprocess.Popen([sys.executable, '-m', 'app.app'], cwd=project_root)
    print('Сервер запущен на http://127.0.0.1:8050/. Откройте ссылку в браузере.')
    print('Чтобы остановить сервер, выполните ячейку ниже.')

### Остановка сервера

После просмотра приложения выполните ячейку ниже, чтобы корректно завершить процесс Dash. Это освободит порт и предотвратит "залипание" фоновых процессов.

In [ ]:
if 'app_process' in globals():
    if app_process.poll() is None:
        app_process.terminate()
        app_process.wait()
        print('Сервер остановлен.')
    else:
        print('Процесс сервера уже завершён.')
else:
    print('Переменная app_process не определена: сервер ещё не запускался в этой сессии.')

## Проверка работы парсеров

Папка `data/parsers/` содержит базовые классы для будущих парсеров внешних источников данных. Ячейка ниже создаёт демонстрационный парсер, наследующийся от `BaseParser`, и показывает, что метод `parse()` возвращает нормализованный результат.

> **Как проверить свои парсеры:** замените реализацию `DemoParser.fetch()` на свою и убедитесь, что словарь `payload` содержит ожидаемые записи. При необходимости добавьте автоматические тесты в каталог `tests/`.

In [ ]:
from data.parsers.base import BaseParser

class DemoParser(BaseParser):
    source = 'demo-source'

    def fetch(self):
        yield {'id': 1, 'title': 'First record', 'value': 42}
        yield {'id': 2, 'title': 'Second record', 'value': 1337}

parser = DemoParser()
result = parser.parse()
print(result)
print('
Полученные записи:')
for record in result.payload['records']:
    print('-', record)